In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

In [ ]:
# Part 1: Loading Images and Finding SIFT Matches

img1, img2, _ = data.stereo_motorcycle()
img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
img2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

matcher = cv2.BFMatcher()
matches = matcher.knnMatch(des1, des2, k=2)

good = []
for m,n in matches:
    if m.distance < 0.75*n.distance:
        good.append(m)

print("Matches:", len(good))

In [ ]:
img_matches = cv2.drawMatches(
    img1, kp1,
    img2, kp2,
    good[:100],
    None
)

plt.figure(figsize=(15,8))
plt.imshow(img_matches)
plt.axis("off")

In [ ]:
# Part 2: Extracting Correspondences

pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
pts2 = np.float32([kp2[m.trainIdx].pt for m in good])

pts1_h = np.column_stack([pts1, np.ones(len(pts1))])
pts2_h = np.column_stack([pts2, np.ones(len(pts2))])

In [ ]:
# Part 3: Custom 8-Point Algorithm

def build_A(pts1_h, pts2_h):

    A = []

    for p1, p2 in zip(pts1_h, pts2_h):

        u,v,_ = p1
        up,vp,_ = p2

        A.append([
            u*up,
            u*vp,
            u,
            v*up,
            v*vp,
            v,
            up,
            vp,
            1
        ])

    return np.asarray(A)

# Solving SVD
A = build_A(pts1_h, pts2_h)
_, _, Vt = np.linalg.svd(A)
F = Vt[-1].reshape(3,3)
print(F)

# Forcing Rank-2
U,S,Vt = np.linalg.svd(F)
S[-1] = 0
F_rank2 = U @ np.diag(S) @ Vt

In [ ]:
# Part 4: Calculating Epipolar Errors
# The Sampson error is a first-order Taylor approximation of the true reprojection error.

def sampson_error(F, pts1_h, pts2_h):

    Fx2 = (F @ pts2_h.T).T
    Ftx1 = (F.T @ pts1_h.T).T

    numerator = np.sum(
        pts1_h * (F @ pts2_h.T).T,
        axis=1
    )**2

    denominator = (
        Fx2[:,0]**2 +
        Fx2[:,1]**2 +
        Ftx1[:,0]**2 +
        Ftx1[:,1]**2
    )

    return np.mean(numerator / denominator)

In [ ]:
# Part 5: Normalized 8-Point

def normalize_points(pts):

    mean = np.mean(pts, axis=0)

    centered = pts - mean

    d = np.mean(
        np.sqrt(np.sum(centered**2,axis=1))
    )

    s = np.sqrt(2)/d

    T = np.array([
        [s,0,-s*mean[0]],
        [0,s,-s*mean[1]],
        [0,0,1]
    ])

    pts_h = np.column_stack([
        pts,
        np.ones(len(pts))
    ])

    pts_n = (T @ pts_h.T).T

    return pts_n, T

# Normalize
pts1_n, T1 = normalize_points(pts1)
pts2_n, T2 = normalize_points(pts2)

# Solving SVD
A = build_A(pts1_n, pts2_n)
_,_,Vt = np.linalg.svd(A)
Fn = Vt[-1].reshape(3,3)

# Forcing Rank-2
U,S,Vt = np.linalg.svd(Fn)
S[-1]=0
Fn = U @ np.diag(S) @ Vt

# Un-normalize
Fn = T1.T @ Fn @ T2

In [ ]:
# Compute error
print("8-point:")
print(sampson_error(F_rank2,
                      pts1_h,
                      pts2_h))

print("Normalized:")
print(sampson_error(Fn,
                      pts1_h,
                      pts2_h))

In [ ]:
# Part 6: RANSAC

# repeat:
#     sample 8 matches randomly
#     estimate F
#     count inliers (error < threshold)
#
# choose model with most inliers
# recompute F using all inliers
#
# Result:
# robust estimation despite incorrect matches

F_ransac, mask = cv2.findFundamentalMat(
    pts1,
    pts2,
    cv2.FM_RANSAC,
    1.0,
    0.99
)

# Inlier
inliers = mask.ravel() == 1
print("Inliers:",
      np.sum(inliers))



In [ ]:
lines1 = cv2.computeCorrespondEpilines(
    pts2[inliers].reshape(-1,1,2),
    2,
    F_ransac
)

lines1 = lines1.reshape(-1,3)

img1_color = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)

for r in lines1[:20]:

    a,b,c = r

    x0 = 0
    y0 = int(-c/b)

    x1 = img1.shape[1]
    y1 = int(-(c+a*x1)/b)

    cv2.line(
        img1_color,
        (x0,y0),
        (x1,y1),
        (0,255,0),
        1
    )

for p in pts1[inliers][:20]:

    cv2.circle(
        img1_color,
        tuple(np.int32(p)),
        5,
        (255,0,0),
        -1
    )

plt.figure(figsize=(10,8))
plt.imshow(cv2.cvtColor(img1_color, cv2.COLOR_BGR2RGB))
plt.axis("off")

In [ ]:
pts1_in_h = pts1_h[inliers]
pts2_in_h = pts2_h[inliers]

print("sampson_error:")
print("8-point:")
print(
    sampson_error(
        F_rank2,
        pts1_in_h,
        pts2_in_h
    )
)

print("Normalized:")
print(
    sampson_error(
        Fn,
        pts1_in_h,
        pts2_in_h
    )
)

print("RANSAC:")
print(
    sampson_error(
        F_ransac,
        pts1_in_h,
        pts2_in_h
    )
)